# Error analysis

Join prediction errors to the validation texts and generate category labels.

In [2]:
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

from src.contracts import validate_prediction_frame
from src.error_analysis import collect_misclassifications, tag_error_category, summarize_error_categories

prediction_path = PROJECT_ROOT / 'results' / 'predictions' / 'model.csv'
if not prediction_path.exists():
    raise FileNotFoundError(f'Prediction file not found: {prediction_path}. Run the classical baseline or provide Person A predictions first.')
val = pd.read_csv(PROJECT_ROOT / 'data' / 'validation.tsv', sep='\t', header=None, names=['uid', 'text_clean', 'label'])
val['text_clean'] = val['text_clean'].fillna('').astype(str)
predictions = pd.read_csv(prediction_path)
validate_prediction_frame(predictions)
text_frame = val[['uid', 'text_clean']]
errors = collect_misclassifications(predictions, text_frame)
errors['category'] = errors.apply(tag_error_category, axis=1)
figures_dir = PROJECT_ROOT / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)
errors.to_csv(figures_dir / 'misclassified_examples.csv', index=False)
summarize_error_categories(errors).to_csv(figures_dir / 'error_categories.csv', index=False)
errors.head()


,uid,y_true,y_pred,p_neg,p_neu,p_pos,text_clean,category
0,27137,1,0,0.558846,0.357859,0.083295,rassi jal gaya lekin aithan nahi gaya .. @ rah...,negation
1,576,1,2,0.172808,0.236135,0.591056,super hilari true realiti noora haramkhor fami...,emoji-only signal
2,9510,1,2,0.085047,0.433070,0.481883,54 ahsan bhai meri taraf se best wish di jye g...,neutral/negative confusion
3,23853,0,1,0.246230,0.453001,0.300769,y'all rememb cute boy follow ig bich like two ...,neutral/negative confusion
4,19075,2,1,0.263441,0.403380,0.333179,esi baat prr madam ek debat apk sath ka hojay ...,spelling variation
